<a href="https://colab.research.google.com/github/YoungSong99/demonhacks_26/blob/main/age_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/Colab_Notebooks/YourTomorrow

# install micromamba
!wget -qO micromamba.tar.bz2 https://micro.mamba.pm/api/micromamba/linux-64/latest
!tar -xvjf micromamba.tar.bz2

!export MAMBA_ROOT_PREFIX=/usr/local/micromamba # under here, venv created

# create venv with python 3.10
!./bin/micromamba env create -f py310_age_env.yml -n age_env -y


!./bin/micromamba run -n age_env pip install fastapi uvicorn nest_asyncio pyngrok

!pip list

Mounted at /content/drive
/content/drive/MyDrive/Colab_Notebooks/YourTomorrow
info/files
info/has_prefix
info/index.json
info/test/run_test.sh
info/hash_input.json
info/recipe/C_ARES_LICENSE.txt
info/licenses/C_ARES_LICENSE.txt
info/recipe/conda_build_config.yaml
info/licenses/REPROC_LICENSE.txt
info/recipe/REPROC_LICENSE.txt
info/licenses/NLOHMANN_JSON_LICENSE.txt
info/recipe/NLOHMANN_JSON_LICENSE.txt
info/recipe/CURL_LICENSE.txt
info/licenses/CURL_LICENSE.txt
info/recipe/CPP_FILESYSTEM_LICENSE.txt
info/recipe/ZLIB_LICENSE.txt
info/licenses/ZLIB_LICENSE.txt
info/licenses/LIBNGHTTP2_LICENSE.txt
info/recipe/LIBNGHTTP2_LICENSE.txt
info/paths.json
info/recipe/LIBLZ4_LICENSE.txt
info/licenses/LIBLZ4_LICENSE.txt
info/recipe/SPDLOG_LICENSE.txt
info/licenses/SPDLOG_LICENSE.txt
info/licenses/FMT_LICENSE.txt
info/recipe/FMT_LICENSE.txt
info/licenses/LIBSOLV_LICENSE.txt
info/recipe/LIBSOLV_LICENSE.txt
info/licenses/mamba/LICENSE
info/recipe/build.sh
info/licenses/ZSTD_LICENSE.txt
info/recipe/TER

In [2]:
!./bin/micromamba run -n age_env pip list

!./bin/micromamba run -n age_env python - <<'PY'

import torch, os
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda:", torch.version.cuda)
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))

Package                  Version     Build
------------------------ ----------- -----
accelerate               0.28.0
annotated-doc            0.0.4
annotated-types          0.7.0
anyio                    4.12.1
asttokens                3.0.1
certifi                  2026.2.25
charset-normalizer       3.4.4
click                    8.3.1
cmake                    4.2.3
contourpy                1.3.2
cycler                   0.12.1
decorator                5.2.1
diffusers                0.27.1
exceptiongroup           1.3.1
executing                2.2.1
fastapi                  0.135.0
filelock                 3.24.3
fonttools                4.61.1
fsspec                   2026.2.0
h11                      0.16.0
hf-xet                   1.3.2
huggingface-hub          0.20.3
idna                     3.11
importlib_metadata       8.7.1
ipython                  8.38.0
jedi                     0.19.2
Jinja2                   3.1.6
kiwisolver               1.4.9
lit                      18.

In [36]:
%%writefile app.py
import io
import base64
import os, uuid
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse, Response, JSONResponse
from age_modifier import AgeModifier

app = FastAPI()

BASE_DIR = "."
AGING_DIR = os.path.join(BASE_DIR, "aging")
MODEL_DIR = os.path.join(BASE_DIR, "models")
DUMMY_PLY_PATH = os.path.join(BASE_DIR, "gaussians.ply")

@app.post("/api/age")
async def age_endpoint(
    file: UploadFile = File(...),
    age: int = Form(...),
    gender: str = Form(...),
    current_age: int = Form(...),
):
    image_id = str(uuid.uuid4())
    data = await file.read()

    modifier = AgeModifier(
        image_bytes=data,
        init_age=current_age,
        target_age=age,
        gender=gender,
    )
    result_img = modifier.generate_age_img()

    # save to the disk
    os.makedirs(AGING_DIR, exist_ok=True)
    save_path = os.path.join(AGING_DIR, f"{image_id}.png")
    result_img.save(save_path, format="PNG")

    return JSONResponse({
        "image_id": image_id,
        "received_age": age,
        "bytes": len(data),
    })

@app.get("/api/images/{image_id}")
def api_get_image(image_id: str):
    candidates = [fn for fn in os.listdir(AGING_DIR) if fn.startswith(image_id)]
    if not candidates:
        return Response(status_code=404)

    path = os.path.join(AGING_DIR, candidates[0])
    media = "image/png" if path.lower().endswith(".png") else "image/jpeg"
    return FileResponse(path, media_type=media)


Overwriting app.py


In [37]:
!./bin/micromamba run -n age_env pip install python-multipart

In [38]:
!pip install fastapi uvicorn nest_asyncio pyngrok
!ngrok config add-authtoken

from pyngrok import ngrok
public_url = ngrok.connect(8000)
print(public_url)
!./bin/micromamba run -n age_env uvicorn app:app --host 127.0.0.1 --port 8000 --app-dir .

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
NgrokTunnel: "https://unwarming-totemic-artie.ngrok-free.dev" -> "http://localhost:8000"
INFO:     Started server process [29099]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
Loading pipeline components...: 100% 6/6 [00:05<00:00,  1.15it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, 